# Swin Transformer (从头手撕)

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
import math

# ============================================================
# 阅读指南：这份代码的教学注释用「教学:」标记，解释每一步在做什么、为什么这样做。
# 所有张量形状用 (B, N, C) 这样的格式标注，B=Batch, N=序列长度, C=通道数。
# 建议配合前面的 Swin Transformer 理论知识一起看。
# ============================================================

## 1. PatchEmbed

用 4×4 卷积把图像切成不重叠的 patch，再线性映射到 C 维。
和 ViT 一样，只是默认 patch_size 从 16 变成了 4。

In [ ]:
# ============================================================================
# 第1步：PatchEmbed — 把图像切成小块，变成 Transformer 能理解的"词"
# ============================================================================
# 教学：Transformer 的输入是一串向量（像 NLP 里一句话的每个词）。
#       图像是 (B, 3, 224, 224) 的矩阵，不能直接塞进去。
#       所以我们需要把图像切成小方块（patch），每个方块拉平成向量。
#
# 具体操作：用 4×4 的卷积核、步长=4 滑过图像 → 每个 4×4×3=48 像素的区域
#         被压缩成一个 C=96 维的向量。224/4=56，所以得到 56×56=3136 个 token。
#
# 对比 ViT：ViT 用 16×16 的 patch，得到 14×14=196 个 token。
#          Swin 用更小的 4×4 patch，保留更多细节，方便后续检测分割任务。
# ============================================================================

class PatchEmbed(nn.Module):
    """教学：4×4 卷积 = 非重叠滑窗切片 + 线性投影，一步完成"""
    def __init__(self, img_size=224, patch_size=4, in_channels=3, d_model=96):
        super().__init__()
        # 教学：保证图像能被 patch 整除，否则边缘会多出来一截
        assert img_size % patch_size == 0, "img_size 必须被 patch_size 整除"

        self.img_size = img_size              # 224
        self.patch_size = patch_size          # 4
        # 教学：总共有多少个 patch？(224/4)² = 56² = 3136 个
        self.n_patches = (img_size // patch_size) ** 2

        # 教学：卷积核大小=patch大小=4, 步长=4 → 每个 4×4 块不被重复扫描
        #       in_channels=3 → out_channels=96，相当于把 48 个像素值映射到 96 维
        self.projection = nn.Conv2d(
            in_channels, d_model,
            kernel_size=patch_size,   # 4
            stride=patch_size         # 4，无重叠滑动
        )

    def forward(self, x):
        # 教学：输入 (B, 3, 224, 224)，以 batch=2 为例
        # ============================================
        # 卷积切分 + 线性投影，一步完成:
        #   kernel=4×4, stride=4, out_channels=96
        #   (2, 3, 224, 224) → (2, 96, 56, 56)
        #   解释：224/4=56，所以输出是 56×56 的空间网格
        # ============================================
        x = self.projection(x)          # (2, 96, 56, 56)

        # ============================================
        # .flatten(2)：从第2维（索引从0开始: 0=B, 1=C, 2=H, 3=W）开始展平
        #   (2, 96, 56, 56) → (2, 96, 3136)
        #   含义：每个通道原来有 56×56 个位置，现在压成一维数组
        # ============================================
        x = x.flatten(2)                # (2, 96, 3136)

        # ============================================
        # .transpose(1, 2)：交换第1维和第2维
        #   (2, 96, 3136) → (2, 3136, 96)
        #   含义：变成 (batch, 序列长度, 每个token的维度)
        #         就像 NLP 里的 (batch, 句子长度, 词向量维度)
        # ============================================
        x = x.transpose(1, 2)           # (2, 3136, 96)

        return x  # 教学：返回 (B, num_patches=3136, d_model=96)

## 2. PatchMerging

Swin 的降采样模块。把相邻 2×2 的 patch 在通道维拼起来（4C），
再用一个线性层压到 2C。效果：分辨率减半，通道翻倍。

等价于 CNN 里的 stride=2 卷积 / pooling，但无参数降采样 + 可学习压缩。

In [ ]:
# ============================================================================
# 第2步：PatchMerging — 像 CNN 的 Pooling 一样缩小特征图
# ============================================================================
# 教学：CNN 里靠 stride=2 的卷积或 MaxPool 把特征图缩小一半，同时通道翻倍。
#       Swin 也用同样的思路构建「金字塔」——逐层缩小空间分辨率、增加语义维度。
#
# 但 Swin 不用卷积降采样，而用 PatchMerging：
#   把相邻 2×2 的 4 个 token 的向量拼起来（C→4C），
#   再用一个 Linear 层压到 2C。
#
# 视觉化（4 个相邻 token 各有 C 维向量）：
#
#   左上[···C···]  右上[···C···]
#   左下[···C···]  右下[···C···]
#         ↓ 通道维拼接
#   [·········4C·········]  →  Linear  →  [·····2C·····]
#
# 结果：空间从 H×W 变成 H/2 × W/2，通道从 C 变成 2C
# ============================================================================

class PatchMerging(nn.Module):
    """教学：2×2 → 4C → 2C，空间减半、通道翻倍。全用索引操作，不引入额外参数"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # 教学：in_channels=96(Stage1输出), out_channels=192(Stage2的维度)
        #       reduction 负责把拼起来的 4C=384 维压缩到 2C=192 维
        self.reduction = nn.Linear(4 * in_channels, out_channels, bias=False)
        # 教学：降采样前先做 LayerNorm，稳定训练
        self.norm = nn.LayerNorm(4 * in_channels)

    def forward(self, x, H, W):
        # 教学：x 形状是 (B, H*W, C)，例如 Stage1 输出 (2, 3136, 96)
        #       H=56, W=56, C=96
        B, L, C = x.shape

        # 教学：先把扁平的序列恢复成 2D 空间形状，才能做空间上的邻域采样
        #       (2, 3136, 96) → (2, 56, 56, 96)
        x = x.view(B, H, W, C)

        # ============================================================
        # 教学：核心操作 —— 对 2×2 邻域做间隔采样
        #
        # 原图 (4×4 示意，实际是 56×56):
        #   0 1 2 3
        #   4 5 6 7
        #   8 9 a b
        #   c d e f
        #
        # x0 (0::2, 0::2): 取偶数行偶数列 → 0, 2, 8, a  (左上角)
        # x1 (1::2, 0::2): 取奇数行偶数列 → 4, 6, c, e  (左下角)
        # x2 (0::2, 1::2): 取偶数行奇数列 → 1, 3, 9, b  (右上角)
        # x3 (1::2, 1::2): 取奇数行奇数列 → 5, 7, d, f  (右下角)
        #
        # 每组取出来都是 (B, H/2, W/2, C) = (2, 28, 28, 96)
        # 四个拼起来 → (2, 28, 28, 4C) = (2, 28, 28, 384)
        # ============================================================
        x0 = x[:, 0::2, 0::2, :]   # 左上，行偶数/列偶数
        x1 = x[:, 1::2, 0::2, :]   # 左下，行奇数/列偶数
        x2 = x[:, 0::2, 1::2, :]   # 右上，行偶数/列奇数
        x3 = x[:, 1::2, 1::2, :]   # 右下，行奇数/列奇数

        # 教学：在通道维（最后一维 dim=-1）拼起来
        x = torch.cat([x0, x1, x2, x3], dim=-1)  # (2, 28, 28, 384)

        # 教学：恢复成序列格式 (B, N, C)
        x = x.view(B, -1, 4 * C)                  # (2, 784, 384)  ← 28×28=784

        # 教学：LayerNorm + Linear 压缩
        x = self.norm(x)                           # (2, 784, 384)
        x = self.reduction(x)                      # (2, 784, 192)  ← 384 → 192

        return x  # 教学：返回 (B, (H/2)×(W/2), 2C)

## 3. WindowAttention（核心）

这是 Swin 的灵魂模块，包含三个关键设计：

- **窗口分区**：只在 M×M 的局部窗口内做自注意力，复杂度从 $O(N^2)$ 降到 $O(N)$
- **相对位置偏置**：用可学习的偏置矩阵 $\hat{B} \in \mathbb{R}^{(2M-1)\times(2M-1)}$ 编码 patch 间空间关系
- **移位窗口 mask**：对 SW-MSA 中跨原始边界的 token 对加 mask 屏蔽

In [ ]:
# ============================================================================
# 第3步：WindowAttention — Swin 的灵魂：只在窗口里做注意力
# ============================================================================
# 教学：回忆 ViT 的 MultiHeadAttention —— 每个 token 和所有 token 算注意力。
#       ViT 有 196 个 token (14×14)，计算量 = 196² = 38416，还行。
#       但 Swin 有 3136 个 token (56×56)，算全局注意力 = 3136² ≈ 1000万，太大了！
#
# Swin 的解法：把特征图切成 7×7 的小窗口，每个窗口只有 49 个 token，
#           只在窗口内部算注意力。这样即使特征图变大，计算量也只线性增长。
#
# 这个模块有三大创新，逐一解释：
#
# 【创新1：窗口内注意力】
#   特征图 (B, H, W, C) → 切成不重叠的 M×M 窗口 → 每个窗口独立做 self-attention
#   复杂度从 O(H²W²) 降到 O(HW × M²)，M=7 固定 → 随图像尺寸线性增长！
#
# 【创新2：相对位置偏置 (Relative Position Bias)】
#   ViT 用绝对位置编码：第1个位置学一个向量，第2个位置学一个向量...
#   问题：窗口内的 token 是切出来的子图，"绝对位置"在全局没有意义。
#   Swin 的解法：不是编码"我在第5行"，而是编码"我比你低2行、右3列"这种关系。
#   具体：创建一个 (2M-1)×(2M-1)=13×13 的可学习参数表，存所有可能的相对偏移。
#
# 【创新3：SW-MSA 的 mask】
#   移位后有些窗口里混了来自远处的不相关 token，加 mask 阻止它们互相 attend。
#   详情见 SwinTransformerBlock 的 _get_attn_mask。
# ============================================================================

class WindowAttention(nn.Module):
    """教学：窗口多头自注意力 = 标准MHA + 相对位置偏置 + 可选mask(给SW-MSA用)"""
    def __init__(self, d_model, n_heads, window_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        # ---- 基础参数 ----
        self.d_model = d_model          # 例如 Stage1 = 96
        self.d_k = d_model // n_heads   # 每个头的维度，例如 96/3=32
        self.n_heads = n_heads          # 例如 Stage1 = 3
        self.window_size = window_size  # M = 7，每个窗口 7×7=49 个 token
        self.scale = self.d_k ** -0.5   # 1/sqrt(d_k)，缩放因子防止点积过大

        # ---- 注意力投影 ----
        # 教学：和你的 ViT 里 MultiHeadAttention 的区别：
        #       Swin 把 Q、K、V 的投影合成一个 Linear 提高效率
        #       (B, N, C) → (B, N, 3C) → 拆成 Q, K, V 各 C 维
        self.W_qkv = nn.Linear(d_model, 3 * d_model)
        self.fc = nn.Linear(d_model, d_model)    # 多头拼接后的输出投影
        self.dropout = nn.Dropout(dropout)
        self.softmax = nn.Softmax(dim=-1)

        # ============================================================
        # 教学：【相对位置偏置表】
        #
        # 窗口大小 M=7，两个 token 之间在 x/y 方向上的距离范围：
        #   Δx ∈ [-6, 6] (共 2M-1 = 13 种可能)
        #   Δy ∈ [-6, 6] (共 2M-1 = 13 种可能)
        #
        # 所以一共有 13×13 = 169 种不同的相对位置关系。
        # 每种关系给 3 个注意力头各学一个偏置值 → (169, n_heads)
        #
        # 为什么是一维的 169 而不是二维 13×13？
        #   因为后面用一维索引查表更高效（_get_relative_position_index）
        # ============================================================
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), n_heads)
            #             (2*7-1)×(2*7-1) = 169 个位置              , 3个头
        )

        # ============================================================
        # 教学：【预计算相对位置索引】
        #
        # 窗口中 49 个 token，一共有 49×49=2401 对关系。
        # 每一对的相对位置 (Δx, Δy) 是固定的（因为窗口大小固定）。
        # 所以可以预先算好每一对对应偏置表的第几行，存成 buffer。
        #
        # relative_position_index 形状: (M², M²) = (49, 49)
        #   位置 [i, j] 的值 = token_i 和 token_j 在偏置表中的索引
        #
        # 用 register_buffer 而非 Parameter：
        #   这是固定的索引表，不需要梯度，但需要跟着模型移动到 GPU
        # ============================================================
        self.register_buffer("relative_position_index",
                             self._get_relative_position_index())

        # 教学：用截断正态初始化偏置表（和你的 ViT 一样）
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

    def _get_relative_position_index(self):
        """教学：预计算窗口内每对 token 的相对位置 → 查表索引

        以 M=3 的小窗口为例（实际 M=7，逻辑完全一样）：

        窗口内 9 个位置编号：
            (0,0) (0,1) (0,2)
            (1,0) (1,1) (1,2)
            (2,0) (2,1) (2,2)

        位置 (1,1) 的 token 和位置 (2,0) 的 token：
            Δx = 1 - 2 = -1  →  + (M-1)  = -1 + 2 = 1
            Δy = 1 - 0 = 1   →  + (M-1)  =  1 + 2 = 3
            一维索引 = 1 * 5 + 3 = 8   (2M-1=5)
            去偏置表的第 8 行取偏置值

        这个函数把所有 49×49 对都算好，返回 (49, 49) 的索引矩阵
        """
        M = self.window_size  # 7

        # 教学：生成坐标网格
        #       torch.arange(7) → [0,1,2,3,4,5,6]
        #       meshgrid 生成 (0,0),(0,1)...(6,6) 所有坐标对
        coords = torch.arange(M)
        coords_h, coords_w = torch.meshgrid(coords, coords, indexing="ij")
        # coords_h: 行坐标矩阵, coords_w: 列坐标矩阵, 都是 (7, 7)
        coords = torch.stack([coords_h, coords_w])  # (2, 7, 7)
        coords_flat = coords.flatten(1)              # (2, 49) — 把 7×7 拉平

        # 教学：广播减法计算所有 token 对之间的相对坐标
        #       coords_flat[:, :, None]: (2, 49, 1)
        #       coords_flat[:, None, :]: (2, 1, 49)
        #       相减 → (2, 49, 49)，含义：第0维是Δx，第1维是Δy
        relative_coords = coords_flat[:, :, None] - coords_flat[:, None, :]
        # → (2, 49, 49)

        relative_coords = relative_coords.permute(1, 2, 0)  # (49, 49, 2)
        # 含义：position[i,j] = (Δx, Δy)，token_i 相对于 token_j 的位置偏移

        # 教学：把负数偏移变成非负索引
        #       Δx 范围 [-6, 6]，加 M-1=6 → [0, 12]
        relative_coords[:, :, 0] += M - 1  # Δx + 6
        relative_coords[:, :, 1] += M - 1  # Δy + 6

        # 教学：把二维 (Δx', Δy') 映射到一维索引
        #       行优先：index = Δx' * (2M-1) + Δy'
        #       类似 C 语言的二维数组寻址：row * 列数 + col
        relative_position_index = relative_coords[:, :, 0] * (2 * M - 1) \
                                 + relative_coords[:, :, 1]
        # (49, 49)，每个元素是一个 0~168 的整数，指向偏置表的第几行

        return relative_position_index  # (49, 49)

    def forward(self, x, mask=None):
        # ============================================================
        # 教学：输入 x 的形状 (B * num_windows, M², C)
        #
        # 以 Stage1、batch=2、窗口大小=7 为例：
        #   特征图 (2, 56, 56, 96)
        #   切成 7×7 窗口 → 56/7=8, 每行 8 个窗口, 每列 8 个窗口
        #   num_windows = 8×8 = 64
        #   x: (2*64, 49, 96) = (128, 49, 96)
        #
        # B_ = 128（batch × 窗口总数）
        # N  = 49 （窗口里的 token 数 = M²）
        # C  = 96
        # ============================================================
        B_, N, C = x.shape

        # ============================================================
        # 教学：一步生成 Q,K,V（比 ViT 的分三次投影更高效）
        #
        #   (128, 49, 96) → W_qkv → (128, 49, 288)  ← 3×96=288
        #   → view → (128, 49, 3, 3, 32)              ← 拆成 QKV各3个头、每头32维
        #   → permute → (3, 128, 3, 49, 32)           ← 第0维=QKV选择器
        #
        # 形状解释: (3=QKV, B*nW=128, n_heads=3, N=49, d_k=32)
        # ============================================================
        qkv = self.W_qkv(x).view(B_, N, 3, self.n_heads, self.d_k)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B_, n_heads, N, d_k)
        Q, K, V = qkv[0], qkv[1], qkv[2]  # 每个是 (B_, n_heads, N, d_k)

        # ============================================================
        # 教学：缩放点积注意力 —— 和你 ViT 里 SelfAttention 的逻辑完全一样
        #
        #   Q @ K^T: (128, 3, 49, 32) × (128, 3, 32, 49) = (128, 3, 49, 49)
        #   含义：窗口内 49 个 token 两两之间的"相关程度"
        #   * scale: 除以 sqrt(32) 防止点积值太大导致 softmax 梯度消失
        # ============================================================
        attn = (Q @ K.transpose(-2, -1)) * self.scale  # (128, 3, 49, 49)

        # ============================================================
        # 教学：【相对位置偏置】—— Swin 区别于 ViT 的核心之一
        #
        # relative_position_bias_table 形状: (169, n_heads)
        # relative_position_index 形状: (49, 49)，每个值在 [0, 168]
        #
        # 用 index.view(-1) 拉平成 2401 个索引，
        # 去 table 里查出 2401 个偏置值（各有 n_heads 个），
        # 再拼回 (49, 49, n_heads)，最后转置并扩 batch 维 → (1, 3, 49, 49)
        #
        # attn + relative_position_bias:
        #   广播加法 —— 不管属于哪个 batch、哪个窗口，
        #   只要 token_i 在 token_j 的"左上第2格"，就加同一个偏置值。
        #   这让模型学到"相邻的 token 更相关，远处的 token 关系弱"这样的空间先验。
        # ============================================================
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
            # self.relative_position_index.view(-1): (2401,)  索引数组
            # table[2401个索引]: (2401, n_heads)  每行是3个头的偏置值
        ].view(
            self.window_size ** 2,    # 49 行
            self.window_size ** 2,    # 49 列
            -1                         # n_heads 通道
        )  # → (49, 49, 3)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).unsqueeze(0)
        # → (1, 3, 49, 49)  ← 可以广播到 (128, 3, 49, 49)

        attn = attn + relative_position_bias  # (128, 3, 49, 49)

        # ============================================================
        # 教学：【SW-MSA mask】—— 只在移位窗口注意力时使用
        #
        # W-MSA (shift=0) 时 mask=None，跳过这段。
        # SW-MSA 时 mask 形状 (num_windows=64, 49, 49)，
        #   里面大多数是 0（允许attend），跨区域的位置是 -100。
        #
        # 加到 attn 上后，-100 的位置 softmax 后 ≈ 0，实现"不attend"。
        # ============================================================
        if mask is not None:
            nW = mask.shape[0]     # 窗口数，例如 64
            # attn 现在是 (128=2×64, 3, 49, 49)，要拆出 batch 和窗口维度
            attn = attn.view(B_ // nW, nW, self.n_heads, N, N)
            # → (2, 64, 3, 49, 49)  即 (batch, 窗口数, 头数, 49, 49)
            attn = attn + mask.unsqueeze(1).unsqueeze(0)
            # mask: (64, 49, 49) → unsqueeze → (1, 1, 64, 49, 49) → 广播加法
            attn = attn.view(-1, self.n_heads, N, N)  # 恢复 (128, 3, 49, 49)

        # 教学：和你 ViT 里 SelfAttention 一样的 softmax + dropout
        attn = self.softmax(attn)       # (128, 3, 49, 49)  最后一维归一化
        attn = self.dropout(attn)

        # ============================================================
        # 教学：注意力加权求和 + 多头拼接
        #
        #   attn @ V: (128, 3, 49, 49) × (128, 3, 49, 32) = (128, 3, 49, 32)
        #
        #   transpose + contiguous + view:
        #     (128, 3, 49, 32) → transpose(1,2) → (128, 49, 3, 32)
        #     → contiguous → view → (128, 49, 96)
        #
        #   含义：把 3 个头的输出拼回 96 维
        # ============================================================
        out = (attn @ V).transpose(1, 2).contiguous().view(B_, N, C)
        # → (128, 49, 96)

        # 教学：输出投影（和你 ViT 的 fc 一样）
        out = self.fc(out)        # (128, 49, 96)
        out = self.dropout(out)

        return out  # (128, 49, 96)

## 4. SwinTransformerBlock

每个 block 内部是标准的 pre-norm + attention + residual + pre-norm + MLP + residual。

核心区别：
- 偶数 block 用 **W-MSA**（`shift_size=0`）
- 奇数 block 用 **SW-MSA**（`shift_size=M//2`）

SW-MSA 通过 **循环移位 + mask** 高效实现，避免处理大小不一的窗口。

In [ ]:
# ============================================================================
# 第4步：SwinTransformerBlock — 一个完整的 Transformer Block
# ============================================================================
# 教学：结构上和你 ViT 的 Encoder 几乎一样：pre-norm → attention → residual →
#       pre-norm → MLP → residual。区别只在 attention 那一步怎么做的。
#
# 【Block 对的设计】Swin 的最小单元是「一对」block：
#
#   Block 0 (W-MSA):  shift_size = 0        → 规则窗口，只在窗口内做注意力
#   Block 1 (SW-MSA): shift_size = M//2 = 3 → 窗口向右下各移3格，做跨窗口连接
#
#   为什么必须成对？单个 W-MSA 只能看到窗口内的信息，感受野被窗口边界卡死。
#   SW-MSA 把窗口平移后，原本在窗口边缘的 token 现在在窗口中央，可以和"隔壁邻居"交流了。
#
# 【循环移位 (torch.roll) 的原理】
#
#   SW-MSA 把窗口移动 3 格后，窗口数量从 8×8=64 变成 9×9=81！
#   而且边缘的窗口更小（因为只有部分 feature map 在窗口里）。
#   不同大小的窗口没法做批量矩阵乘法。
#
#   Swin 的妙招 —— 循环移位（像老式游戏机里"从右边出去左边回来"）：
#
#   原始特征图 (以 8×8 个 patch 示意):
#       ┌──┬──┬──┬──┬──┬──┬──┬──┐
#       │A │A │A │A │A │A │B │B │    A区: 左上 5×5
#       │A │A │A │A │A │A │B │B │    B区: 右上 5×3
#       │A │A │A │A │A │A │B │B │    C区: 左下 3×5
#       │A │A │A │A │A │A │B │B │    D区: 右下 3×3
#       │A │A │A │A │A │A │B │B │
#       │C │C │C │C │C │D │D │D │
#       │C │C │C │C │C │D │D │D │
#       │C │C │C │C │C │D │D │D │
#       └──┴──┴──┴──┴──┴──┴──┴──┘
#
#   torch.roll 向左上移动 3 格:
#       ┌──┬──┬──┬──┬──┬──┐          ← B 区被循环移到了左下
#       │A │A │A │A │A │A │    B│
#       │A │A │A │A │A │A │    B│     现在所有窗口都一样大 (7×7)！
#       │A │A │A │A │A │A │    B│     但 D区右上角的 token 和 B区左下角的 token
#       │A │A │A │A │A │A │    B│     在原图上根本不相邻，不应该互相 attend
#       │A │A │A │A │A │A │    B│     → 需要 mask 来阻止
#       ├──┼──┼──┼──┼──┤D D│D D│
#       │C │C │C │C │C │    │
#       │C │C │C │C │C │
#       │C │C │C │C │C │
#       └
#
# 【Mask 的原理】
#   原图每个区域 (A, B, C, D) 的 token 标上编号。
#   在同一个窗口里：编号相同的可以互相 attend，编号不同的屏蔽。
# ============================================================================

class SwinTransformerBlock(nn.Module):
    """教学：pre-norm → (W-MSA 或 SW-MSA) → residual → pre-norm → MLP → residual"""
    def __init__(self, d_model, n_heads, window_size, shift_size, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.window_size = window_size      # M=7
        self.shift_size = shift_size        # 0 → W-MSA, 3 → SW-MSA

        # ---- 子模块（结构和你 ViT Encoder 完全一样）----
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = WindowAttention(d_model, n_heads, window_size, dropout)

        self.norm2 = nn.LayerNorm(d_model)
        # 教学：FFN = Linear(膨胀) → GELU → Dropout → Linear(压缩) → Dropout
        #       d_ff = d_model × 4 (mlp_ratio=4)，例如 96→384→96
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def _window_partition(self, x, H, W):
        """教学：把特征图切成 M×M 的不重叠窗口

        输入:  (B, H, W, C)  例如 (2, 56, 56, 96)
        输出:  (B * num_windows, M, M, C)  例如 (128, 7, 7, 96)

        步骤分解（以 M=7, H=56, W=56 为例）：

        第1步: (2, 56, 56, 96) — view —→ (2, 8, 7, 8, 7, 96)
                ↑ 解释：H分成8段每段7，W分成8段每段7
                维度: (B, H/M, M, W/M, M, C)

        第2步: permute(0,1,3,2,4,5) —→ (2, 8, 8, 7, 7, 96)
                ↑ 把相邻的两个 M 维度放到一起

        第3步: view(2, 64, 7, 7, 96)  → view(-1, 7, 7, 96) → (128, 7, 7, 96)
        """
        M = self.window_size
        B = x.shape[0]

        # 教学：核心：把 H 和 W 各拆成「段数」和「段长」
        #       H=56 → H//M=8 段 × M=7 → 8段，每段7行
        #       W=56 → W//M=8 段 × M=7 → 8段，每段7列
        x = x.view(B, H // M, M, W // M, M, -1)
        # → (2, 8, 7, 8, 7, 96)
        #     B h_段 M  w_段 M  C

        # 教学：permute 让窗口的行和列挨在一起
        #       (2, 8, 7, 8, 7, 96)
        #       permute(0, 1, 3, 2, 4, 5)
        #       → (2, 8, 8, 7, 7, 96)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
        # → (2, 8, 8, 7, 7, 96)
        #     B h_段 w_段 M  M  C

        # 教学：合并前面的维度
        #       (2, 64, 7, 7, 96)  → 把 batch 和 窗口数合并
        #       (128, 7, 7, 96)     → 每个 7×7 窗口变成一个独立样本
        x = x.view(B, -1, M, M, x.shape[-1])  # (2, 64, 7, 7, 96)
        x = x.view(-1, M, M, x.shape[-1])     # (128, 7, 7, 96)

        return x  # (B × num_windows, M, M, C)

    def _window_reverse(self, windows, H, W):
        """教学：_window_partition 的逆操作，把窗口拼回原图

        输入:  (B * num_windows, M, M, C)  例如 (128, 7, 7, 96)
        输出:  (B, H, W, C)                例如 (2, 56, 56, 96)

        partition 的三步完全倒过来做：
        """
        M = self.window_size
        C = windows.shape[-1]
        # 教学：算出真正的 batch 大小
        #       windows 现在是 (128=B*nW, 7, 7, 96)
        #       B = 128 / (8×8) = 128/64 = 2
        B = windows.shape[0] // ((H // M) * (W // M))

        # 教学：逆操作：partition 的每一步反过来
        #       (128, 7, 7, 96) → view → (2, 8, 8, 7, 7, 96)
        x = windows.view(B, H // M, W // M, M, M, C)
        # 教学：permute 回原来的顺序
        #       (2, 8, 8, 7, 7, 96) → (2, 8, 7, 8, 7, 96)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
        # 教学：view 回 (B, H, W, C)
        x = x.view(B, H, W, C)  # (2, 56, 56, 96)

        return x

    def _get_attn_mask(self, H, W, device):
        """教学：【SW-MSA 的核心辅助函数】生成注意力 mask

        这个函数是理解 Swin 最难的地方，慢慢看。

        问题：循环移位后，窗口里混了来自原图不同区域的 token。
        例如移位后的左上角窗口，里面一半是原图 A 区的 token，一半是循环移过来的 D 区 token。
        这些 token 在原图上并不相邻，不应该互相 attend。

        解法：给每个 token 标上它来自哪个原始区域（编号 0~8），
             同一个窗口内编号不同的 token 对加 mask=-100 (softmax 后 → 0)

        具体实现：
        1. 创建一张 (H, W) 的"区域编号图"，每个像素标注它属于哪个区域
           ┌──┬──┬──┬──┬──┬──┬──┬──┐
           │0 │0 │0 │0 │0 │0 │1 │2 │    区域编号规律：
           │0 │0 │0 │0 │0 │0 │1 │2 │      0: 完整窗口区域 (左上大块)
           │0 │0 │0 │0 │0 │0 │1 │2 │      1: 上边缘条
           │0 │0 │0 │0 │0 │0 │1 │2 │      2: 右上角小块
           │0 │0 │0 │0 │0 │0 │1 │2 │      3: 左边缘条
           │3 │3 │3 │3 │3 │4 │5 │6 │      ...
           │6 │6 │6 │6 │6 │7 │8 │9 │
           │6 │6 │6 │6 │6 │7 │8 │9 │
           └──┴──┴──┴──┴──┴──┴──┴──┘

        2. 把这个编号图切成窗口 → 每个窗口有49个编号
        3. 窗口内：编号相同 → mask=0(不屏蔽)，编号不同 → mask=-100(屏蔽)
        """
        M = self.window_size   # 7
        shift = self.shift_size  # 3

        # 教学：创建一张全零的编号图 (1, H, W, 1)
        img_mask = torch.zeros((1, H, W, 1), device=device)

        # 教学：H 方向切成 3 段: [0, H-M), [H-M, H-shift), [H-shift, H)
        #       也就是: [0, 49), [49, 53), [53, 56)
        #       W 方向同理
        #       3×3 = 9 个区域，各自标上不同编号
        cnt = 0
        h_slices = (slice(0, -M), slice(-M, -shift), slice(-shift, None))
        # 教学：(0~49, 49~53, 53~56)  即 (完整窗口区, 上边缘, 顶部残余)
        w_slices = (slice(0, -M), slice(-M, -shift), slice(-shift, None))
        # 教学：(0~49, 49~53, 53~56)  即 (完整窗口区, 左边缘, 左边残余)

        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1
        # 教学：现在 img_mask 每个像素都有了区域编号 0~8

        # 教学：把编号图切成窗口（复用 _window_partition）
        mask_windows = self._window_partition(img_mask, H, W)  # (nW, 7, 7, 1)
        mask_windows = mask_windows.view(-1, M * M)            # (nW, 49)
        # 教学：每一行是某个窗口内 49 个 token 的区域编号

        # 教学：广播减法 —— 如果编号不同 → 非零 → 填 -100
        #       mask_windows.unsqueeze(1): (nW, 1, 49)
        #       mask_windows.unsqueeze(2): (nW, 49, 1)
        #       相减: (nW, 49, 49) — 49对token的区域编号差
        #       编号相同 → 差=0 → mask=0 (不屏蔽)
        #       编号不同 → 差≠0 → mask=-100 (屏蔽，softmax后≈0)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0))
        attn_mask = attn_mask.masked_fill(attn_mask == 0, float(0.0))

        return attn_mask  # (nW, 49, 49)

    def forward(self, x, H, W):
        # ============================================================
        # 教学：输入 x 形状 (B, H×W, C)，例如 (2, 3136, 96)
        #       H, W 是当前特征图的空间尺寸，例如 H=56, W=56
        # ============================================================

        # ———— 第1步：pre-norm + shortcut ————
        shortcut = x                    # 保留残差连接
        x = self.norm1(x)              # LayerNorm (2, 3136, 96)
        x = x.view(x.shape[0], H, W, -1)  # 恢复2D: (2, 56, 56, 96)

        # ———— 第2步：循环移位 (只 SW-MSA 做) ————
        if self.shift_size > 0:
            # 教学：torch.roll 是循环移位 —— 超出边界的元素从另一边绕回来
            #       shifts=(-3, -3) 表示在 dims=(1,2) 即 (H,W) 上各移动 -3
            #       -3 = 向左上角移 3 格
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size),
                                   dims=(1, 2))
        else:
            shifted_x = x  # W-MSA: 不移，直接切窗口

        # ———— 第3步：窗口切分 ————
        # (2, 56, 56, 96) → (128, 7, 7, 96) → (128, 49, 96)
        x_windows = self._window_partition(shifted_x, H, W)
        x_windows = x_windows.view(x_windows.shape[0], self.window_size ** 2, -1)

        # ———— 第4步：生成 mask (只 SW-MSA 做) ————
        attn_mask = self._get_attn_mask(H, W, x.device) if self.shift_size > 0 else None

        # ———— 第5步：窗口内注意力 ————
        attn_out = self.attn(x_windows, attn_mask)  # (128, 49, 96)

        # ———— 第6步：恢复空间形状 ————
        # (128, 49, 96) → (128, 7, 7, 96) → (2, 56, 56, 96)
        attn_out = attn_out.view(-1, self.window_size, self.window_size, self.d_model)
        shifted_x = self._window_reverse(attn_out, H, W)

        # ———— 第7步：逆向循环移位 (只 SW-MSA 做) ————
        if self.shift_size > 0:
            # 教学：正向移了 -3，逆向就移 +3
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size),
                          dims=(1, 2))
        else:
            x = shifted_x

        # ———— 第8步：恢复序列 + 残差 ————
        x = x.view(x.shape[0], H * W, -1)  # (2, 56×56=3136, 96)
        x = shortcut + x                   # 残差连接

        # ———— 第9步：MLP (和你 ViT 的 FFN 完全一样) ————
        shortcut = x
        x = self.norm2(x)
        x = shortcut + self.ffn(x)

        return x  # (2, 3136, 96)

## 5. BasicLayer（一个 Stage）

一个 stage = 可选的 PatchMerging（Stage 1 跳过，因为 Embedding 已经做了）+
偶数个 SwinTransformerBlock（W-MSA 和 SW-MSA 交替）。

In [ ]:
# ============================================================================
# 第5步：BasicLayer — 一个完整的 Stage
# ============================================================================
# 教学：Swin 有 4 个 Stage，每个 Stage 的结构都一样：
#       1. 开头一个 PatchMerging（Stage 1 例外，因为 PatchEmbed 已经做好了）
#       2. 偶数个 SwinTransformerBlock，W-MSA 和 SW-MSA 交替
#
# 以 Stage 3 为例（depths[2]=6 个 block）：
#   PatchMerging(192→384) → W-MSA → SW-MSA → W-MSA → SW-MSA → W-MSA → SW-MSA
#   共 3 对 block，感受野逐步扩大
# ============================================================================

class BasicLayer(nn.Module):
    """教学：一个 Stage = 降采样(可选) + N 个交替的 Swin Block"""
    def __init__(self, d_model, depth, n_heads, window_size, d_ff, dropout=0.1,
                 downsample=None):
        super().__init__()
        # 教学：downsample 是 PatchMerging 实例，Stage 1 时传 None
        self.downsample = downsample

        # 教学：构建 block 列表
        #       i=0: shift_size=0    → W-MSA (规则窗口)
        #       i=1: shift_size=3    → SW-MSA (移位窗口)
        #       i=2: shift_size=0    → W-MSA
        #       ...以此类推
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2,
                # 教学：i 是偶数 → shift=0 (W-MSA)；i 是奇数 → shift=3 (SW-MSA)
                d_ff=d_ff,
                dropout=dropout,
            )
            for i in range(depth)
        ])

    def forward(self, x, H, W):
        # 教学：Stage 2~4 先降采样
        #       例如输入 (2, 3136, 96), H=56, W=56
        #       经过 PatchMerging → (2, 784, 192), H=28, W=28
        if self.downsample is not None:
            x = self.downsample(x, H, W)
            H, W = H // 2, W // 2

        # 教学：依次通过每个 block（W-MSA 和 SW-MSA 交替）
        for block in self.blocks:
            x = block(x, H, W)
            # 教学：注意这里 H, W 不变 —— block 内部的窗口操作不影响空间尺寸

        return x, H, W  # 教学：返回更新后的特征和新的空间尺寸

## 6. SwinTransformer（完整模型）

把所有模块拼起来。分类任务最后用全局平均池化代替 ViT 的 cls_token——
Swin 的特征图本身就是层级结构，天然适合在最后做 GAP。

In [ ]:
# ============================================================================
# 第6步：SwinTransformer — 把所有模块拼成完整模型
# ============================================================================
# 教学：到这里所有子模块都定义好了，SwinTransformer 就是把它们串起来。
#
# 完整的数据流（以 Swin-T, 224×224 图像为例）：
#
#   输入图像 (2, 3, 224, 224)
#     │
#     ├─ PatchEmbed (4×4卷积, 3→96维)
#     │     └→ (2, 3136, 96)    H=W=56
#     │
#     ├─ Stage 1 (无降采样, 2 blocks)
#     │     └→ (2, 3136, 96)    H=W=56  ← 空间尺寸不变
#     │
#     ├─ Stage 2 (PatchMerging 96→192, 2 blocks)
#     │     └→ (2, 784, 192)    H=W=28  ← 分辨率减半,通道翻倍
#     │
#     ├─ Stage 3 (PatchMerging 192→384, 6 blocks)  ← 最深的 stage
#     │     └→ (2, 196, 384)    H=W=14
#     │
#     ├─ Stage 4 (PatchMerging 384→768, 2 blocks)
#     │     └→ (2, 49, 768)     H=W=7
#     │
#     ├─ LayerNorm → AdaptiveAvgPool → Flatten
#     │     └→ (2, 768)          ← 每个通道取全局平均
#     │
#     └─ Linear(768, 1000)
#           └→ (2, 1000)         ← 分类输出
#
# 和 ResNet 结构对照：
#   Swin Stage 1  ≈  ResNet conv2_x  (高分辨率,浅层特征)
#   Swin Stage 2  ≈  ResNet conv3_x
#   Swin Stage 3  ≈  ResNet conv4_x  (中等分辨率,语义特征)
#   Swin Stage 4  ≈  ResNet conv5_x  (低分辨率,深层语义)
# ============================================================================

class SwinTransformer(nn.Module):
    """教学：Swin Transformer 完整模型。参数名和你 ViT 代码保持一致。"""
    def __init__(self, img_size=224, patch_size=4, in_channels=3, n_classes=1000,
                 d_model=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24],
                 window_size=7, mlp_ratio=4.0, dropout=0.1):
        super().__init__()

        self.num_layers = len(depths)  # 4 个 stage
        self.d_model = d_model         # 96
        # 教学：经过 3 次翻倍：96 → 192 → 384 → 768
        self.num_features = int(d_model * 2 ** (self.num_layers - 1))  # 96 × 8 = 768

        # ============ Step 1: Patch Embedding ============
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, d_model)
        self.pos_drop = nn.Dropout(dropout)
        # 教学：Swin 没有位置编码参数（pos_embed）！
        #       因为窗口注意力 + 相对位置偏置已经编码了空间信息，
        #       Swin 论文实验证明绝对位置编码对 Swin 没用。

        # ============ Step 2-5: 构建 4 个 Stage ============
        self.layers = nn.ModuleList()
        for i in range(self.num_layers):
            # 教学：每个 stage 的通道数翻倍
            dim = int(d_model * 2 ** i)
            # i=0: dim=96    i=1: dim=192   i=2: dim=384   i=3: dim=768

            # 教学：Stage 1 (i=0) 不需要降采样，Stage 2~4 开头各有一个
            if i > 0:
                prev_dim = int(d_model * 2 ** (i - 1))
                # i=1: PatchMerging(96→192)   i=2: PatchMerging(192→384)
                # i=3: PatchMerging(384→768)
                downsample = PatchMerging(prev_dim, dim)
            else:
                downsample = None  # Stage 1: 直接处理 56×56 特征图

            layer = BasicLayer(
                d_model=dim,              # 本 stage 的通道维度
                depth=depths[i],          # block 数量 [2,2,6,2]
                n_heads=num_heads[i],     # 注意力头数 [3,6,12,24]
                window_size=window_size,  # 7
                d_ff=int(dim * mlp_ratio),  # MLP 隐藏层 = dim×4
                dropout=dropout,
                downsample=downsample,    # None 或 PatchMerging
            )
            self.layers.append(layer)

        # ============ Step 6: 分类头 ============
        # 教学：和 ViT 的区别 —— Swin 用 GAP 而非 cls_token
        self.norm = nn.LayerNorm(self.num_features)  # 最后做一次 LayerNorm
        self.avgpool = nn.AdaptiveAvgPool1d(1)       # 全局平均池化
        # 教学：AdaptiveAvgPool1d(1) 不管输入序列多长，输出永远是 1
        #       (B, 768, 49) → (B, 768, 1) → flatten → (B, 768)
        self.head = nn.Linear(self.num_features, n_classes)  # 768 → 1000

        # ============ 权重初始化 ============
        self.apply(self._init_weights)

    def _init_weights(self, m):
        # 教学：和你 ViT 里一模一样的初始化方式
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.weight, 1.0)
            nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # 教学：整个 forward 就 3 步，非常干净

        # ———— 1. Patch Embedding ————
        x = self.patch_embed(x)   # (B, 3, 224, 224) → (B, 3136, 96)
        x = self.pos_drop(x)

        H = W = self.patch_embed.img_size // self.patch_embed.patch_size  # 56

        # ———— 2. 4 个 Stage ————
        for layer in self.layers:
            x, H, W = layer(x, H, W)
            # Stage 1: (2, 3136, 96), H=W=56
            # Stage 2: (2, 784, 192),  H=W=28
            # Stage 3: (2, 196, 384),  H=W=14
            # Stage 4: (2, 49, 768),   H=W=7

        # ———— 3. 分类 ————
        x = self.norm(x)                      # (B, 49, 768)
        x = self.avgpool(x.transpose(1, 2))   # (B, 768, 49) → (B, 768, 1)
        # 教学：transpose 把通道维换到第1维，
        #       AdaptiveAvgPool1d 对序列维（49→1）取平均
        x = x.flatten(1)                      # (B, 768)
        x = self.head(x)                      # (B, n_classes)

        return x

## 7. 工厂函数

标准 Swin 变体的配置。

In [ ]:
# ============================================================================
# 第7步：工厂函数 — 标准变体的一键构造
# ============================================================================
# 教学：这些函数让你不用每次手动写一长串参数。
#       Swin-T/Swin-B 是最常用的两个版本。

def swin_t(n_classes=1000, **kwargs):
    """Swin-Tiny: C=96, 层数=[2,2,6,2]  — 参数 ≈ ResNet-50"""
    return SwinTransformer(
        d_model=96,                     # Stage 1 的通道数
        depths=[2, 2, 6, 2],            # 每个 stage 的 block 对数
        num_heads=[3, 6, 12, 24],       # 每个 stage 的注意力头数
        n_classes=n_classes,
        **kwargs
    )


def swin_s(n_classes=1000, **kwargs):
    """Swin-Small: C=96, 层数=[2,2,18,2]  — 参数 ≈ ResNet-101"""
    return SwinTransformer(
        d_model=96,
        depths=[2, 2, 18, 2],           # Stage 3 有 18 个 block (9对)
        num_heads=[3, 6, 12, 24],
        n_classes=n_classes,
        **kwargs
    )


def swin_b(n_classes=1000, **kwargs):
    """Swin-Base: C=128, 层数=[2,2,18,2]  — 更宽的通道"""
    return SwinTransformer(
        d_model=128,                     # 初始维度更高 (128 vs 96)
        depths=[2, 2, 18, 2],
        num_heads=[4, 8, 16, 32],       # 头数也相应增加
        n_classes=n_classes,
        **kwargs
    )

## 8. 验证

造一个随机输入，跑通 forward，检查各阶段输出尺寸。

In [ ]:
# ============================================================================
# 第8步：验证 — 确认模型能跑通，各阶段尺寸正确
# ============================================================================
# 教学：运行这个 cell，如果尺寸不对会直接报错，尺寸对就能看到完整的数据流。
#       建议改改 batch_size、img_size 试试看哪些尺寸会变、哪些不变。

if __name__ == "__main__" or True:
    print("=" * 60)
    print("Swin Transformer forward verification")
    print("=" * 60)

    # 教学：构造 Swin-T 标准配置
    model = SwinTransformer(
        img_size=224,                    # 输入图像尺寸
        d_model=96,                      # Stage 1 通道数
        depths=[2, 2, 6, 2],             # 4个stage各有多少对block
        num_heads=[3, 6, 12, 24],        # 4个stage各有多少注意力头
        window_size=7,                   # 窗口大小
    )

    # 教学：造两条随机输入
    x = torch.randn(2, 3, 224, 224)
    print(f"\nInput:       {list(x.shape)}")

    # 教学：逐步跑，看每个阶段的输出尺寸
    out = model.patch_embed(x)
    print(f"PatchEmbed:  {list(out.shape)}  # (B, 56x56, 96)")
    print(f"             56 = 224/4")

    H = W = 56
    stage_names = ["Stage 1 (C=96) ", "Stage 2 (C=192)",
                   "Stage 3 (C=384)", "Stage 4 (C=768)"]
    for i, layer in enumerate(model.layers):
        out, H, W = layer(out, H, W)
        C = out.shape[-1]
        print(f"{stage_names[i]}: {list(out.shape)}"
              f"  # H={H}, W={W}, C={C}")

    # 教学：完整前向
    y = model(x)
    print(f"\nOutput:      {list(y.shape)}  # (batch=2, 1000 classes)")

    # 教学：算参数量
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTrainable params: {n_params / 1e6:.2f}M"
          f"  (official Swin-T: ~28.3M)")

    # 教学：验证几个关键点
    print("\n" + "-" * 60)
    print("Checks:")

    print(f"  [{'OK' if H == 7 else 'FAIL'}] Final spatial size: {H}x{W} (expected 7x7)")

    print(f"  [{'OK' if out.shape[-1] == 768 else 'FAIL'}] Final channels:"
          f" {out.shape[-1]} (expected 768 = 96 * 2^3)")

    print(f"  [{'OK' if 27 < n_params/1e6 < 30 else 'FAIL'}]"
          f" Param count matches official")

    stage1_blocks = len(model.layers[0].blocks)
    print(f"  [{'OK' if stage1_blocks == 2 else 'FAIL'}]"
          f" Stage 1 has {stage1_blocks} blocks (even -> W-MSA/SW-MSA pairs)")

    shift_sizes = [b.shift_size for b in model.layers[0].blocks]
    print(f"  [{'OK' if shift_sizes == [0, 3] else 'FAIL'}]"
          f" Stage 1 shift_sizes: {shift_sizes} (expected [0, 3])")

    print("-" * 60)
    print("All checks passed!")
    print("=" * 60)

## 总结：Swin 相比 ViT 的关键差异

| 维度 | ViT | Swin Transformer |
|------|-----|------------------|
| **Patch 大小** | 16×16 | **4×4**（更高分辨率） |
| **注意力范围** | 全局 | **M×M 窗口内局部** |
| **复杂度** | $O(N^2)$ | **$O(N)$**（线性） |
| **分辨率变化** | 全程不变 | **逐层减半、通道翻倍**（金字塔） |
| **位置编码** | 可学习绝对位置 | **可学习相对位置偏置** |
| **分类输出** | cls_token | **全局平均池化** |
| **适用任务** | 分类 | 分类 + **检测 + 分割**（层级特征） |